# 08 — Reversible Instance Normalization (RevIN) from Scratch

## The big idea

**RevIN** was introduced by Kim et al. (2022) in *"Reversible Instance Normalization
for Accurate Time-Series Forecasting against Distribution Shift."*

The key insight is simple but powerful:

> Instead of normalizing with global statistics computed once over all training data,
> normalize **each input window independently** using only its own mean and std.
> After the model makes its prediction, **reverse** the normalization to get
> predictions in the original scale.

## Why does this help?

| Problem | Global Norm | RevIN |
|---------|-------------|-------|
| Mean shift | ✗ Model sees wrong baseline | ✓ Each window centred at 0 |
| Variance shift | ✗ Wrong scale | ✓ Each window scaled to unit std |
| New regime | ✗ Out-of-distribution | ✓ Window is always normalized |

## The algorithm

```
NORMALIZE:
  μ  ← mean(window)
  σ  ← std(window) + ε       # ε prevents division by zero
  x̂  ← (window - μ) / σ     # normalized window fed to model

MODEL:
  ŷ_norm ← model(x̂)          # model predicts in normalized space

DENORMALIZE:
  ŷ  ← ŷ_norm × σ + μ       # back to original scale
```

The same μ and σ that were computed in the normalize step are used in
denormalize — this is the "reversible" part.

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from utils.ml_utils import RevIN
from utils.plotting import plot_before_after_distribution, plot_line

## Step 1 — Implement RevIN from scratch

Let's build it from scratch first so the mechanism is crystal clear.

In [ ]:
class RevINScratch:
    """Minimal RevIN for learning purposes."""

    def __init__(self, eps=1e-5):
        self.eps = eps
        self.mean_ = None
        self.std_  = None

    def normalize(self, x):
        self.mean_ = x.mean()               # ① compute instance mean
        self.std_  = x.std() + self.eps     # ② compute instance std
        return (x - self.mean_) / self.std_ # ③ standardize

    def denormalize(self, x_norm):
        return x_norm * self.std_ + self.mean_  # ④ reverse the transform

## Step 2 — Test with a simple window

In [ ]:
window = np.array([150.2, 152.1, 149.8, 153.4, 155.0,
                   154.3, 156.7, 157.2, 155.9, 158.0])

revin = RevINScratch()

# Forward pass
w_norm = revin.normalize(window)

print('Original window :', window)
print(f'Stored mean     : {revin.mean_:.4f}')
print(f'Stored std      : {revin.std_:.4f}')
print('Normalized      :', w_norm.round(4))
print(f'Norm mean       : {w_norm.mean():.6f}  (≈ 0)')
print(f'Norm std        : {w_norm.std():.6f}   (≈ 1)')

In [ ]:
# Simulate model output (slightly noisy normalized predictions)
np.random.seed(7)
y_pred_norm = w_norm[-3:] + np.random.normal(0, 0.1, 3)

# Reverse pass
y_pred_orig = revin.denormalize(y_pred_norm)

print('Predicted (normalized scale):', y_pred_norm.round(4))
print('Predicted (original scale)  :', y_pred_orig.round(4))
print(f'These are around {y_pred_orig.mean():.1f} — same range as the input window!')

## Step 3 — Reversibility proof

In [ ]:
revin2    = RevINScratch()
normed    = revin2.normalize(window)
recovered = revin2.denormalize(normed)

max_error = np.max(np.abs(window - recovered))
print(f'Max reconstruction error: {max_error:.2e}')
print('Perfectly reversible ✓' if max_error < 1e-10 else 'Not perfectly reversible!')

## Step 4 — Distribution shift example

Show how RevIN neutralises a mean shift between two windows.

In [ ]:
np.random.seed(42)

window_regime1 = np.random.normal(100, 5, 20)   # Training regime
window_regime2 = np.random.normal(180, 12, 20)  # New regime (mean shift + var shift)

r1, r2 = RevINScratch(), RevINScratch()
norm1   = r1.normalize(window_regime1)
norm2   = r2.normalize(window_regime2)

print(f'Regime 1 raw  — mean: {window_regime1.mean():.1f}, std: {window_regime1.std():.1f}')
print(f'Regime 2 raw  — mean: {window_regime2.mean():.1f}, std: {window_regime2.std():.1f}')
print()
print(f'Regime 1 norm — mean: {norm1.mean():.4f},  std: {norm1.std():.4f}')
print(f'Regime 2 norm — mean: {norm2.mean():.4f},  std: {norm2.std():.4f}')
print()
print('After RevIN: both windows look the same to the model!')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, data, label, color in zip(
    axes[0],
    [window_regime1, window_regime2],
    ['Regime 1 (raw)', 'Regime 2 (raw)'],
    ['#4C72B0', '#DD8452'],
):
    ax.hist(data, bins=10, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'μ={data.mean():.1f}')
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Value')
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

for ax, data, label, color in zip(
    axes[1],
    [norm1, norm2],
    ['Regime 1 (RevIN)', 'Regime 2 (RevIN)'],
    ['#4C72B0', '#DD8452'],
):
    ax.hist(data, bins=10, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'μ={data.mean():.3f}')
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Normalized value')
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('RevIN normalizes away the regime difference', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5 — Using our library's RevIN

In [ ]:
revin_lib = RevIN(eps=1e-5)
normed    = revin_lib.normalize(window)
recovered = revin_lib.denormalize(normed)

print(f'Stats stored: {revin_lib.stats}')
print(f'Max reconstruction error: {np.max(np.abs(window - recovered)):.2e}')

---
## Conceptual summary

```
Input window (any regime) ──►  RevIN.normalize()  ──► Normalized window
                                  stores μ, σ              ▼
                                                      Forecasting Model
                                                           ▼
Original-scale prediction  ◄── RevIN.denormalize() ◄── Prediction in norm space
```

**The model always sees normalized data regardless of the current price regime.**

---
## Exercises

1. What happens if you call `denormalize()` before `normalize()`? Fix it with a check.
2. Implement RevIN for a **batch** of windows: shape `(n_windows, window_length)`
   where each row gets its own μ and σ.
3. What is the role of `eps`? Try `eps=0` on a constant array and observe.
4. If the model output is a 5-step forecast (not just 1 step), does RevIN still work?
   Why or why not?

---
**Next →** `09_revin_project_stock_forecasting.ipynb` — putting it all together.